# Heston dynamic: Greeks and theta breakeven decomposition

Under Heston, $dS = (r-q)S\,dt + \sqrt{v}\,S\,dW^S$, $dv = \kappa(\bar\theta - v)\,dt + \xi\sqrt{v}\,dW^v$, $d\langle W^S, W^v\rangle = \rho\,dt$.

### Heston parameters

| symbol | code name | definition | role in the dynamics | value used |
|---|---|---|---|---|
| $v_0$ | `v0` | instantaneous variance today ($\sqrt{v_0}$ is the spot volatility) | starting point of the variance process; sets the level of the short-dated vols | 0.01 (10% vol) |
| $\bar\theta$ | `theta` | long-run variance | level that $v$ mean-reverts to; sets the level of the long-dated vols | 0.0625 (25% vol) |
| $\kappa$ | `kappa` | speed of mean reversion (per year) | how fast $v$ moves from $v_0$ to $\bar\theta$; the half-life is $\ln 2/\kappa$ | 8 (half-life ≈ 0.087y ≈ 22 business days) |
| $\xi$ | `xi` | volatility of variance (vol-of-vol) | size of the random shocks to $v$; drives the smile curvature (volga) | 1.2 |
| $\rho$ | `rho` | correlation between the spot and variance Brownian motions | sign and size of the skew (vanna); $\rho<0$: spot down ⇒ vol up ⇒ put skew | −0.75 |
| $r$ | `r` | continuously compounded risk-free rate | discounting and forward drift | 4% |
| $q$ | `q` | continuous dividend yield | forward drift $(r-q)$ | 1% |

How the parameters shape the surface:
- **Term structure.** The expected variance at horizon $s$ is $\mathbb{E}[v_s] = \bar\theta + (v_0 - \bar\theta)e^{-\kappa s}$. $v_0 < \bar\theta$ gives an upward-sloping term structure. A large $\kappa$ makes the slope steep over the 1w–1m segment.
- **Skew.** $\rho < 0$ together with a large $\xi$ gives a pronounced put skew. Its effect is largest at short maturities.
- **Feller condition.** $2\kappa\bar\theta \ge \xi^2$ keeps $v$ strictly positive. Here $2\kappa\bar\theta = 1.0 < \xi^2 = 1.44$, so the condition is violated: $v$ can touch zero, but it is reflected back. This does not affect the pricing, which uses the characteristic function.
- **Units.** All parameters are annualised on a 252-business-day year (see *Business-day time* below).
- **Notation.** $\bar\theta$ (code: `theta`) is the long-run variance. It is not the Greek $\Theta$ (code: `theta` column in `df`).

Every option price $V(t, S, v)$ satisfies the pricing PDE:

$$
\Theta + \tfrac12 v S^2\,\Gamma + \rho\,\xi\,v\,S\,\text{Vanna} + \tfrac12 \xi^2 v\,\text{Volga} + \kappa(\bar\theta - v)\,\text{Vega} + (r-q)\,S\,\Delta - r\,V = 0
$$

Rearranged, **theta is exactly paid for by five breakeven terms**:

$$
\boxed{\;\Theta = \underbrace{-\tfrac12 v S^2 \Gamma}_{\text{be\_gamma}}
\;\underbrace{-\,\kappa(\bar\theta - v)\,\text{Vega}}_{\text{be\_vega}}
\;\underbrace{-\,\rho\,\xi\,v\,S\,\text{Vanna}}_{\text{be\_vanna}}
\;\underbrace{-\,\tfrac12 \xi^2 v\,\text{Volga}}_{\text{be\_volga}}
\;\underbrace{+\,r V - (r-q) S \Delta}_{\text{be\_rho}}\;}
$$

Each term is the expected P&L that the matching risk earns over $dt$ when the delta-hedged book follows the model dynamics:

| term | P&L source | realised quantity it breaks even against |
|---|---|---|
| be_gamma | $\tfrac12\Gamma\,(dS)^2$ | spot variance $v S^2 dt$ |
| be_vanna | $\text{Vanna}\,dS\,dv$ | spot/variance covariance $\rho\xi v S\,dt$ |
| be_volga | $\tfrac12\text{Volga}\,(dv)^2$ | variance of variance $\xi^2 v\,dt$ |
| be_vega | $\text{Vega}\,\mathbb{E}[dv]$ | drift of variance $\kappa(\bar\theta - v)\,dt$ |
| be_rho | financing of the delta-hedged position | $r(V - S\Delta) + qS\Delta$ |

**Corrections to the original list**
1. **be_vega** is not a vol-move breakeven. In Heston, variance has a deterministic drift $\kappa(\bar\theta - v)$, and holding vega earns that drift. This term matters most when $v_0$ is far from $\bar\theta$ (steep term structure).
2. **be_rho** does not come from the rho Greek $\partial V/\partial r$. It is the **carry/financing** of the delta-hedged position, $rV - (r-q)S\Delta$, so it needs the price, the delta and the dividend yield $q$. Rho is still reported as a Greek.
3. **Vega, vanna and volga must be taken w.r.t. the instantaneous variance $v$** (the Heston state variable), not Black-Scholes implied vol. Otherwise the identity does not hold.
4. **Theta is $\partial V/\partial t$ with $S$ and $v$ held fixed** (pure time decay). It must be in the same time unit as the breakevens. Here both are **per business day**.

Delta strikes use the Black-Scholes spot delta evaluated at the Heston implied vol of each strike. The ATM point is the 50-delta call.

### Business-day time
All time is measured in **business days**: $\tau = \text{busdays}(\text{valuation}, \text{expiry}) / 252$, with weekends and an optional holiday list excluded.
The Heston parameters, $r$ and $q$ are therefore annualised on a 252-business-day year. One-day theta and breakevens are the annual rates divided by 252.

### Theta and breakeven over a horizon of $N$ business days
Theta accelerates as expiry approaches, so $N \times$ one-day theta is not the $N$-day decay. With $h = \min(N/252,\ \tau)$ and $S$, $v$ frozen:

$$
\Theta_{h} = V(\tau - h) - V(\tau) = \int_{\tau-h}^{\tau} \Theta(s)\,ds
= \sum_{i \,\in\, \{\Gamma,\ \text{vega},\ \text{vanna},\ \text{volga},\ \rho\}} \underbrace{\int_{\tau-h}^{\tau} \text{be}_i(s)\,ds}_{\text{be}_{i,h}}
$$

Each horizon breakeven is the instantaneous breakeven integrated along the decay path, so the identity $\Theta_h = \sum_i \text{be}_{i,h}$ holds exactly.
If the option expires inside the horizon, the horizon stops at expiry ($V(0)$ = intrinsic).
The integral uses Gauss-Legendre in $u = \sqrt{s}$, which absorbs the $1/\sqrt{s}$ blow-up of gamma at expiry.
Pricing uses the COS method, whose truncation range scales with $\sqrt{\tau}$, so the near-expiry nodes are priced accurately.

### Forward formulation of Heston
With constant $r$ and $q$, the forward to the option expiry $T$ is $F_t = S_t\,e^{(r-q)(T-t)}$. Under the pricing measure it has no drift:

$$
dF = \sqrt{v}\,F\,dW^F, \qquad dv = \kappa(\bar\theta - v)\,dt + \xi\sqrt{v}\,dW^v, \qquad d\langle W^F, W^v\rangle = \rho\,dt
$$

The variance process and the parameters $(v_0, \bar\theta, \kappa, \xi, \rho)$ are unchanged. Let $C(\tau, F, v) = e^{r\tau}V$ be the **undiscounted (forward) premium**. It satisfies a PDE with no $r$ or $q$ terms:

$$
-\frac{\partial C}{\partial \tau} + \tfrac12 v F^2 C_{FF} + \rho\,\xi\,v\,F\,C_{Fv} + \tfrac12 \xi^2 v\,C_{vv} + \kappa(\bar\theta - v)\,C_v = 0
$$

Define the forward Greeks in present-value terms, with $D = e^{-r\tau}$:
- $\Gamma^F = D\,C_{FF}$, $\text{Vanna}^F = D\,C_{Fv}$, $\text{Vega}^F = D\,C_v$, $\text{Volga}^F = D\,C_{vv}$
- **forward theta** $\Theta^F = -D\,\partial C/\partial\tau\big|_{F,v}$: the decay of the forward premium with $F$ and $v$ held fixed, discounted to today

The breakeven identity then has **four terms and no carry term**:

$$
\boxed{\;\Theta^F = \underbrace{-\tfrac12 v F^2 \Gamma^F}_{\text{be\_gamma}}
\;\underbrace{-\,\kappa(\bar\theta - v)\,\text{Vega}^F}_{\text{be\_vega}}
\;\underbrace{-\,\rho\,\xi\,v\,F\,\text{Vanna}^F}_{\text{be\_vanna}}
\;\underbrace{-\,\tfrac12 \xi^2 v\,\text{Volga}^F}_{\text{be\_volga}}\;}
$$

**Link to the spot formulation.** Since $\partial F/\partial S = e^{(r-q)\tau}$, we have $F^2\Gamma^F = S^2\Gamma$, $F\,\text{Vanna}^F = S\,\text{Vanna}$, $\text{Vega}^F = \text{Vega}$ and $\text{Volga}^F = \text{Volga}$. The four breakevens are therefore identical in both formulations. Only theta differs, by the carry:

$$
\Theta = \Theta^F + \underbrace{rV - (r-q)\,S\,\Delta}_{\text{be\_rho}}
$$

Holding $F$ fixed moves the carry out of theta and into the forward rolling toward spot. $\Theta^F$ is the "pure theta" that many desks quote. In the code, the forward premium is priced directly as $C(\tau, F, v)$: the Heston price with spot $= F$ and $r = q = 0$.

### Monte Carlo path and parameter estimation
We simulate **one path** of $(S_t, v_t)$ and treat the spot and the instantaneous variance as observed once per business day ($\Delta = 1/252$). From this path we back out $\bar\theta$, $\kappa$, $\xi$ and $\rho$.

**Simulation.** The variance is sampled from its exact CIR transition (a scaled non-central $\chi^2$) on fine sub-steps. This keeps $v \ge 0$ exactly, which matters here because the Feller condition is violated. Log-spot uses the exact relation $\int\sqrt{v}\,dW^v = \big(v_{t+\delta} - v_t - \kappa\bar\theta\,\delta + \kappa\int v\,ds\big)/\xi$:

$$
\Delta\ln S = (r-q)\,\delta - \tfrac12\!\int\! v\,ds + \frac{\rho}{\xi}\Big(v_{t+\delta} - v_t - \kappa\bar\theta\,\delta + \kappa\!\int\! v\,ds\Big) + \sqrt{(1-\rho^2)\!\int\! v\,ds}\;Z
$$

The integral $\int v\,ds$ is approximated with the trapezoid rule on each sub-step.

**Estimators** (sampled once per business day):
1. **Mean reversion: $\kappa$, $\bar\theta$.** The CIR conditional mean is linear: $\mathbb{E}[v_{t+\Delta}\,|\,v_t] = \bar\theta(1-e^{-\kappa\Delta}) + e^{-\kappa\Delta}v_t$. Regress $v_{t+\Delta} = a + b\,v_t + \varepsilon^v$ by OLS. Then
$\hat\kappa = -\ln \hat b/\Delta$ and $\hat{\bar\theta} = \hat a/(1-\hat b)$.
2. **Vol of vol: $\xi$.** Match the residual variance to the CIR conditional variance:
$\hat\xi^2 = \dfrac{\sum (\varepsilon^v)^2}{\sum \Big[v_t\,\dfrac{e^{-\kappa\Delta}-e^{-2\kappa\Delta}}{\kappa} + \bar\theta\,\dfrac{(1-e^{-\kappa\Delta})^2}{2\kappa}\Big]}$, evaluated at $\hat\kappa, \hat{\bar\theta}$. This is $\approx \sum(\varepsilon^v)^2 / \sum v_t\Delta$.
3. **Correlation: $\rho$.** Take the spot residual $\varepsilon^S = \Delta\ln S - (r - q - \tfrac12 v_t)\Delta$. Both residuals scale with $\sqrt{v_t}$, so $\hat\rho = \text{corr}(\varepsilon^S, \varepsilon^v)$.

**What to expect.**
- $\xi$ and $\rho$ come from the quadratic variation and covariation of the path. They become more precise as the sampling frequency increases, so a single path pins them down well.
- $\kappa$ and $\bar\theta$ come from the drift. Their precision depends only on the **length** of the path in years, not the sampling frequency. Roughly, $\text{sd}(\hat\kappa) \approx \sqrt{2\kappa/T_{\text{path}}}$. The OLS estimate of $\kappa$ is also biased upwards in finite samples. Expect a large error on $\kappa$ even with 20 years of data.
- **Measure.** On real data, $\kappa$ and $\bar\theta$ estimated from a historical path are real-world ($\mathbb{P}$) parameters. They differ from the pricing ($\mathbb{Q}$) parameters by the variance risk premium. $\xi$ and $\rho$ are the same under both measures. Here the path is simulated with the pricing parameters, so the true values are those in the Input section.

### Forward implied volatility: 3W in 1W, formula vs exact Heston forward start
Take $t$ = 1W expiry and $T$ = 1M expiry. Strictly, the forward period is $T - t$ = 17 business days ≈ 3W. For each delta strike, compare two forward vols:

**1. Forward-variance formula**, applied delta by delta to the spot implied vols at the same delta:

$$
\sigma_{\text{fwd}}(t, T) = \sqrt{\frac{T\,\sigma(0,T)^2 - t\,\sigma(0,t)^2}{T - t}}
$$

**2. Exact Heston forward start.** The option fixes its strike at $t$ as $k\,S_t$ and pays $(S_T/S_t - k)^+$ at $T$. At $t$ it is a vanilla Heston option on $S = 1$ with strike $k$, tenor $\tau = T - t$, and variance $v_t$, which is random today. Its value at $t$, per unit of $S_t$, averaged over $v_t$, is

$$
\mathbb{E}^{\mathbb{Q}}\big[\,C_H(1, k, \tau;\ v_t)\,\big], \qquad v_t = c\,\chi'^2_{d}(\lambda),\quad c = \frac{\xi^2(1 - e^{-\kappa t})}{4\kappa},\quad d = \frac{4\kappa\bar\theta}{\xi^2},\quad \lambda = \frac{v_0\,e^{-\kappa t}}{c}
$$

using the exact CIR law of $v_t$. The expectation is computed on 400 equally-weighted quantiles. The forward implied vol is the Black-Scholes vol (on $S = 1$, strike $k$, tenor $\tau$, rates $r$, $q$) that reproduces this premium. $k$ is solved so that the option has the target BS spot delta at its own implied vol, the same delta convention as the spot strikes.

The formula only sees the two spot smiles. The exact price depends on the whole distribution of $v_t$. In Heston the skew scales roughly as $\rho\xi/\sqrt{v}$. Because $v$ drifts up from $v_0$ toward $\bar\theta$, the forward smile is flatter than the formula implies.

*Convention note.* With the payoff $(S_T - k S_t)^+$ instead, the expectation is taken under the share measure. There $v$ has parameters $\kappa^* = \kappa - \rho\xi$ and $\bar\theta^* = \kappa\bar\theta/(\kappa - \rho\xi)$, which gives a slightly different forward smile.

## Import

In [95]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import brentq, least_squares
from scipy.stats import ncx2, norm

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## Custom functions

In [96]:
BDAYS_PER_YEAR = 252.0   # every year fraction in the notebook is business days / 252
COS_N, COS_L = 512, 16.0  # COS expansion terms and truncation width (in standard deviations)


def year_fraction_bd(start, end, holidays=()):
    """Business-day year fraction between two dates (weekends + holidays excluded)."""
    return np.busday_count(np.datetime64(start.date()), np.datetime64(end.date()), holidays=list(holidays)) / BDAYS_PER_YEAR


def roll_following(date, holidays=()):
    """Roll a date forward to the next business day if it falls on a weekend / holiday."""
    return pd.Timestamp(np.busday_offset(np.datetime64(date.date()), 0, roll="following", holidays=list(holidays)))


def heston_cf(u, T, v0, kappa, theta, xi, rho):
    """Characteristic function of X_T = ln(S_T/S_0) - (r-q)T under Heston ('little trap' form)."""
    iu = 1j * u
    b = kappa - rho * xi * iu
    d = np.sqrt(b * b + xi**2 * (iu + u * u))
    g = (b - d) / (b + d)
    e = np.exp(-d * T)
    C = kappa * theta / xi**2 * ((b - d) * T - 2.0 * np.log((1.0 - g * e) / (1.0 - g)))
    D = (b - d) / xi**2 * (1.0 - e) / (1.0 - g * e)
    return np.exp(C + D * v0)


def heston_price(S, K, T, r, q, v0, kappa, theta, xi, rho, cp):
    """Heston European price with the COS method (Fang-Oosterlee).
    The truncation range scales with sqrt(T), so the pricer stays accurate down to minutes before expiry.
    K and cp can be arrays; cp = +1 call, -1 put. T <= 0 returns the intrinsic value."""
    K, cp = np.atleast_1d(K).astype(float), np.atleast_1d(cp)
    if T <= 0:
        return np.maximum(cp * (S - K), 0.0)
    x = np.log(S / K)[:, None]
    sd = np.sqrt(max(v0, theta) * T)
    c1 = (r - q) * T - 0.5 * sd**2
    a, b = c1 - COS_L * sd, c1 + COS_L * sd                    # range for Z = ln(S_T/S_0)
    w = np.arange(COS_N) * np.pi / (b - a)
    phi = heston_cf(w, T, v0, kappa, theta, xi, rho) * np.exp(1j * w * (r - q) * T - 1j * w * a)
    phi[0] *= 0.5

    def chi(c, d):  # int_c^d e^z cos(w(z-a)) dz
        return (np.cos(w * (d - a)) * np.exp(d) - np.cos(w * (c - a)) * np.exp(c)
                + w * np.sin(w * (d - a)) * np.exp(d) - w * np.sin(w * (c - a)) * np.exp(c)) / (1.0 + w * w)

    def psi(c, d):  # int_c^d cos(w(z-a)) dz
        ws = np.where(w == 0, 1.0, w)
        return np.where(w == 0, d - c, (np.sin(w * (d - a)) - np.sin(w * (c - a))) / ws)

    z0 = np.clip(-x, a, b)                                     # exercise boundary in Z
    call_coef = np.exp(x) * chi(z0, b) - psi(z0, b)
    put_coef = psi(a, z0) - np.exp(x) * chi(a, z0)
    coef = np.where(cp[:, None] > 0, call_coef, put_coef) * 2.0 / (b - a)
    return K * np.exp(-r * T) * (np.real(phi) * coef).sum(axis=1)


def bs_price(S, K, T, r, q, sigma, cp):
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return cp * (S * np.exp(-q * T) * norm.cdf(cp * d1) - K * np.exp(-r * T) * norm.cdf(cp * d2))


def bs_delta(S, K, T, r, q, sigma, cp):
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return cp * np.exp(-q * T) * norm.cdf(cp * d1)


def implied_vol(price, S, K, T, r, q, cp):
    f = lambda s: bs_price(S, K, T, r, q, s, cp) - price
    try:
        return brentq(f, 1e-4, 5.0, xtol=1e-12)
    except ValueError:  # price outside BS bounds (numerical noise far OTM)
        return np.nan


def delta_to_strike(target_delta, S, T, r, q, hp, cp):
    """Strike whose BS spot delta, computed with the Heston implied vol at that strike, equals target_delta."""
    width = 5.0 * np.sqrt(max(hp["v0"], hp["theta"]) * T)

    def f(K):
        price = heston_price(S, K, T, r, q, cp=cp, **hp)[0]
        iv = implied_vol(price, S, K, T, r, q, cp)
        if np.isnan(iv):  # deep OTM / ITM limit of the BS delta
            iv = 1e-4
        return bs_delta(S, K, T, r, q, iv, cp) - target_delta

    return brentq(f, S * np.exp(-width), S * np.exp(width), xtol=1e-10)


def heston_greeks(S, K, T, r, q, hp, cp, hS=None, hv=None, hT=1e-5, hr=1e-4):
    """Central finite-difference Greeks, time in business-day years.
    Vega / vanna / volga are w.r.t. the instantaneous variance v. Theta is per (business) year."""
    v0 = hp["v0"]
    hS = hS or min(1e-4 * S, 1e-2 * S * np.sqrt(v0 * T))   # bump shrinks with the spot std near expiry
    hv = hv or 1e-2 * v0
    hT = min(hT, 0.1 * T)
    P = lambda s=S, t=T, rr=r, v=v0: heston_price(s, K, t, rr, q, **{**hp, "v0": v}, cp=cp)[0]

    V = P()
    Vsu, Vsd = P(s=S + hS), P(s=S - hS)
    Vvu, Vvd = P(v=v0 + hv), P(v=v0 - hv)
    delta = (Vsu - Vsd) / (2 * hS)
    gamma = (Vsu - 2 * V + Vsd) / hS**2
    vega = (Vvu - Vvd) / (2 * hv)
    volga = (Vvu - 2 * V + Vvd) / hv**2
    vanna = (P(s=S + hS, v=v0 + hv) - P(s=S + hS, v=v0 - hv)
             - P(s=S - hS, v=v0 + hv) + P(s=S - hS, v=v0 - hv)) / (4 * hS * hv)
    theta = -(P(t=T + hT) - P(t=T - hT)) / (2 * hT)   # dV/dt = -dV/dT
    rho_ = (P(rr=r + hr) - P(rr=r - hr)) / (2 * hr)
    return dict(price=V, delta=delta, gamma=gamma, vega=vega, vanna=vanna, volga=volga, theta=theta, rho=rho_)


def heston_breakevens(g, S, r, q, hp):
    """Split theta into the Heston PDE terms (per year). Signed so that the sum equals theta."""
    v, kappa, theta_, xi, rho = hp["v0"], hp["kappa"], hp["theta"], hp["xi"], hp["rho"]
    return dict(
        be_gamma=-0.5 * v * S**2 * g["gamma"],
        be_vega=-kappa * (theta_ - v) * g["vega"],
        be_vanna=-rho * xi * v * S * g["vanna"],
        be_volga=-0.5 * xi**2 * v * g["volga"],
        be_rho=r * g["price"] - (r - q) * S * g["delta"],
    )


def theta_breakeven_horizon(S, K, T, r, q, hp, cp, horizon_bd, n_nodes=40):
    """Theta over `horizon_bd` business days and the breakevens that pay for it (S and v frozen).

    theta_h = V(T - h) - V(T)  =  integral over the horizon of the instantaneous theta
            = sum_i integral over the horizon of be_i(tau) d tau
    If the option expires inside the horizon, the horizon is cut at expiry (V(0) = intrinsic).
    The integral uses Gauss-Legendre in u = sqrt(tau), which absorbs the 1/sqrt(tau) blow-up of gamma at expiry."""
    h = min(horizon_bd / BDAYS_PER_YEAR, T)
    theta_h = (heston_price(S, K, T - h, r, q, cp=cp, **hp) - heston_price(S, K, T, r, q, cp=cp, **hp))[0]

    x, w = np.polynomial.legendre.leggauss(n_nodes)
    u_lo, u_hi = np.sqrt(T - h), np.sqrt(T)
    u = 0.5 * (u_hi - u_lo) * x + 0.5 * (u_hi + u_lo)
    w = 0.5 * (u_hi - u_lo) * w * 2.0 * u               # d tau = 2u du
    be_h = dict.fromkeys(["be_gamma", "be_vega", "be_vanna", "be_volga", "be_rho"], 0.0)
    for tau, wi in zip(u**2, w):
        be = heston_breakevens(heston_greeks(S, K, tau, r, q, hp, cp), S, r, q, hp)
        for key in be_h:
            be_h[key] += wi * be[key]
    return {"theta": theta_h, **be_h}


def heston_forward_premium(F, K, T, hp, cp):
    """Undiscounted (forward) Heston premium C(T, F, v) = E[payoff]: the Heston price with spot = F and r = q = 0."""
    return heston_price(F, K, T, 0.0, 0.0, cp=cp, **hp)


def heston_greeks_fwd(F, K, T, r, hp, cp, hF=None, hv=None, hT=1e-5):
    """Forward-formulation Greeks in present-value terms (D = e^{-rT} times derivatives of C(T, F, v)).
    Theta is -D dC/dT with F and v held fixed, per (business) year."""
    v0 = hp["v0"]
    hF = hF or min(1e-4 * F, 1e-2 * F * np.sqrt(v0 * T))
    hv = hv or 1e-2 * v0
    hT = min(hT, 0.1 * T)
    D = np.exp(-r * T)
    C = lambda f=F, t=T, v=v0: heston_forward_premium(f, K, t, {**hp, "v0": v}, cp)[0]

    C0 = C()
    Cfu, Cfd = C(f=F + hF), C(f=F - hF)
    Cvu, Cvd = C(v=v0 + hv), C(v=v0 - hv)
    return dict(
        premium=D * C0,
        delta=D * (Cfu - Cfd) / (2 * hF),
        gamma=D * (Cfu - 2 * C0 + Cfd) / hF**2,
        vega=D * (Cvu - Cvd) / (2 * hv),
        volga=D * (Cvu - 2 * C0 + Cvd) / hv**2,
        vanna=D * (C(f=F + hF, v=v0 + hv) - C(f=F + hF, v=v0 - hv)
                   - C(f=F - hF, v=v0 + hv) + C(f=F - hF, v=v0 - hv)) / (4 * hF * hv),
        theta=-D * (C(t=T + hT) - C(t=T - hT)) / (2 * hT),
    )


def heston_breakevens_fwd(g, F, hp):
    """Split the forward theta into the four Heston PDE terms (per year). Signed so that the sum equals theta."""
    v, kappa, theta_, xi, rho = hp["v0"], hp["kappa"], hp["theta"], hp["xi"], hp["rho"]
    return dict(
        be_gamma=-0.5 * v * F**2 * g["gamma"],
        be_vega=-kappa * (theta_ - v) * g["vega"],
        be_vanna=-rho * xi * v * F * g["vanna"],
        be_volga=-0.5 * xi**2 * v * g["volga"],
    )


def simulate_heston_path(S0, r, q, hp, n_years, substeps=20, seed=None):
    """One Heston path sampled once per business day.
    v: exact CIR transition (non-central chi-square) on `substeps` sub-steps per day, so v >= 0 exactly.
    ln S: exact relation int sqrt(v) dW^v = (dv - kappa*theta*dt + kappa*int v)/xi, with a trapezoid rule for int v."""
    rng = np.random.default_rng(seed)
    v0, kappa, theta, xi, rho = hp["v0"], hp["kappa"], hp["theta"], hp["xi"], hp["rho"]
    n = int(round(n_years * BDAYS_PER_YEAR)) * substeps
    dt = 1.0 / (BDAYS_PER_YEAR * substeps)
    e = np.exp(-kappa * dt)
    c = xi**2 * (1.0 - e) / (4.0 * kappa)       # scale of the non-central chi-square
    dof = 4.0 * kappa * theta / xi**2           # degrees of freedom

    v = np.empty(n + 1)
    v[0] = v0
    for i in range(n):
        v[i + 1] = c * rng.noncentral_chisquare(dof, v[i] * e / c)

    int_v = 0.5 * (v[:-1] + v[1:]) * dt
    int_sqrtv_dWv = (v[1:] - v[:-1] - kappa * theta * dt + kappa * int_v) / xi
    dlnS = ((r - q) * dt - 0.5 * int_v + rho * int_sqrtv_dWv
            + np.sqrt((1.0 - rho**2) * int_v) * rng.standard_normal(n))
    lnS = np.log(S0) + np.concatenate([[0.0], np.cumsum(dlnS)])
    return np.exp(lnS[::substeps]), v[::substeps]


def estimate_heston_params(S, v, dt, r, q):
    """Back out (theta, kappa, xi, rho) from one observed path of spot S and instantaneous variance v."""
    v_t, v_next = v[:-1], v[1:]
    # 1) kappa, theta: OLS of v_{t+dt} on v_t (exact CIR conditional mean)
    (a, b), *_ = np.linalg.lstsq(np.column_stack([np.ones_like(v_t), v_t]), v_next, rcond=None)
    kappa = -np.log(b) / dt
    theta = a / (1.0 - b)
    eps_v = v_next - (a + b * v_t)
    # 2) xi: residual variance vs exact CIR conditional variance (per unit xi^2)
    e = np.exp(-kappa * dt)
    cond_var = v_t * (e - e * e) / kappa + theta * (1.0 - e) ** 2 / (2.0 * kappa)
    xi = np.sqrt((eps_v**2).sum() / cond_var.sum())
    # 3) rho: correlation of the spot and variance residuals
    eps_s = np.diff(np.log(S)) - (r - q - 0.5 * v_t) * dt
    rho = np.corrcoef(eps_s, eps_v)[0, 1]
    return dict(theta=theta, kappa=kappa, xi=xi, rho=rho)


def cir_variance_quantiles(v0, t, kappa, theta, xi, n=400):
    """n equally-weighted points of the exact CIR law of v_t given v_0 (mid-point quantiles of a scaled non-central chi-square)."""
    c = xi**2 * (1.0 - np.exp(-kappa * t)) / (4.0 * kappa)
    return c * ncx2.ppf((np.arange(n) + 0.5) / n, 4.0 * kappa * theta / xi**2, v0 * np.exp(-kappa * t) / c)


def heston_forward_start_premium(k, t, tau, r, q, hp, cp, n=400):
    """E^Q[ C_H(S=1, K=k, tau; v_t) ]: value at t, per unit of S_t, of the forward-start option paying (S_T/S_t - k)^+
    (strike fixed at t as k * S_t, expiry T = t + tau). Today's price is e^{-rt} times this."""
    v_t = cir_variance_quantiles(hp["v0"], t, hp["kappa"], hp["theta"], hp["xi"], n)
    return np.mean([heston_price(1.0, k, tau, r, q, cp=cp, **{**hp, "v0": v})[0] for v in v_t])


def forward_start_iv_at_delta(target_delta, t, tau, r, q, hp, cp, n=400):
    """Relative strike k and forward implied vol of the forward-start option whose BS spot delta
    (S=1, K=k, tenor tau, at its own implied vol) equals target_delta."""
    def iv_of(k):
        return implied_vol(heston_forward_start_premium(k, t, tau, r, q, hp, cp, n), 1.0, k, tau, r, q, cp)

    def f(k):
        iv = iv_of(k)
        return bs_delta(1.0, k, tau, r, q, 1e-4 if np.isnan(iv) else iv, cp) - target_delta

    width = 3.5 * np.sqrt(max(hp["v0"], hp["theta"]) * tau)
    k = brentq(f, np.exp(-width), np.exp(width), xtol=1e-8)
    return k, iv_of(k)


HESTON_PARAM_NAMES = ["v0", "theta", "kappa", "xi", "rho"]


def fast_options_calibration(market, x0=None, bounds=None):
    """Calibrate Heston to option PVs and return (fitted parameters, enriched dataframe).

    market columns: strike, time_to_maturity (business-day years), forward, discount, pv,
                    optional cp (+1 call / -1 put; default: out-of-the-money option, call if strike >= forward).
    Objective: least squares on (model - market) undiscounted premium divided by the Black vega,
               i.e. approximately implied-vol errors. Pricing uses Heston homogeneity C(F, K) = F * C(1, K/F),
               so every maturity is priced in one vectorised COS call.
    Output dataframe: input + pv_heston + forward-formulation Greeks (PV terms) + breakevens (per business day)."""
    m = market.reset_index(drop=True).copy()
    if "cp" not in m:
        m["cp"] = np.where(m["strike"] >= m["forward"], 1, -1)
    K, T, F, D, cp = (m[c].to_numpy(float) for c in ["strike", "time_to_maturity", "forward", "discount", "cp"])
    undisc = m["pv"].to_numpy(float) / D

    # Black implied vol and vega of the market quotes -> residual weights
    iv_mkt = np.array([implied_vol(u, f, k, t, 0.0, 0.0, c) for u, f, k, t, c in zip(undisc, F, K, T, cp)])
    d1 = (np.log(F / K) + 0.5 * iv_mkt**2 * T) / (iv_mkt * np.sqrt(T))
    vega_mkt = np.maximum(F * norm.pdf(d1) * np.sqrt(T), 1e-4 * F * np.sqrt(T))
    groups = [np.flatnonzero(T == t) for t in np.unique(T)]

    def model_undisc(x):
        hp = dict(zip(HESTON_PARAM_NAMES, x))
        out = np.empty_like(undisc)
        for idx in groups:
            out[idx] = F[idx] * heston_price(1.0, K[idx] / F[idx], T[idx][0], 0.0, 0.0, cp=cp[idx], **hp)
        return out

    if x0 is None:  # short / long implied variance as v0 / theta, generic values for the rest
        x0 = [np.nanmin(iv_mkt[T == T.min()]) ** 2, np.nanmedian(iv_mkt[T == T.max()]) ** 2, 3.0, 1.0, -0.5]
    if bounds is None:
        bounds = ([1e-4, 1e-4, 0.05, 0.05, -0.999], [1.0, 1.0, 50.0, 5.0, 0.999])
    fit = least_squares(lambda x: (model_undisc(x) - undisc) / vega_mkt, x0, bounds=bounds,
                        x_scale=[0.01, 0.01, 1.0, 0.1, 0.1], xtol=1e-12, ftol=1e-12, gtol=1e-12)
    params = dict(zip(HESTON_PARAM_NAMES, fit.x))

    rows = []
    for k, t, f, d, c in zip(K, T, F, D, cp):
        r_t = -np.log(d) / t
        g = heston_greeks_fwd(f, k, t, r_t, params, int(c))
        be = heston_breakevens_fwd(g, f, params)
        rows.append({
            "pv_heston": g["premium"],
            "delta": g["delta"], "gamma": g["gamma"], "vega": g["vega"], "vanna": g["vanna"], "volga": g["volga"],
            "theta": g["theta"] / BDAYS_PER_YEAR,
            **{key: val / BDAYS_PER_YEAR for key, val in be.items()},
        })
    out = pd.concat([m, pd.DataFrame(rows)], axis=1)
    out["sum_be"] = out[["be_gamma", "be_vega", "be_vanna", "be_volga"]].sum(axis=1)
    return params, out


def fast_parameters_calibration(variance, forward, dt=1.0 / BDAYS_PER_YEAR):
    """Real-world (P) Heston parameters from time series of instantaneous variance and of a forward price.

    variance, forward: aligned arrays / Series sampled every dt (business-day years).
    kappa, theta: OLS of v_{t+dt} on v_t (exact CIR conditional mean);
    xi: residual variance matched to the exact CIR conditional variance;
    rho: correlation between the forward log-return residual and the variance residual.
    The forward drift under P is unknown, so the log-returns are de-meaned (the drift is O(dt), the noise O(sqrt(dt)))."""
    v = np.asarray(variance, float)
    lnF = np.log(np.asarray(forward, float))
    v_t, v_next = v[:-1], v[1:]
    (a, b), *_ = np.linalg.lstsq(np.column_stack([np.ones_like(v_t), v_t]), v_next, rcond=None)
    kappa = -np.log(b) / dt
    theta = a / (1.0 - b)
    eps_v = v_next - (a + b * v_t)
    e = np.exp(-kappa * dt)
    cond_var = v_t * (e - e * e) / kappa + theta * (1.0 - e) ** 2 / (2.0 * kappa)
    xi = np.sqrt((eps_v**2).sum() / cond_var.sum())
    ret = np.diff(lnF) + 0.5 * v_t * dt
    rho = np.corrcoef(ret - ret.mean(), eps_v)[0, 1]
    return dict(theta=theta, kappa=kappa, xi=xi, rho=rho)


def expected_integrated_variance(T, v0, kappa, theta):
    """E[ int_0^T v_s ds ] under Heston/CIR dynamics (closed form)."""
    T = np.asarray(T, float)
    return theta * T + (v0 - theta) * (1.0 - np.exp(-kappa * T)) / kappa


def fast_forward_vol(hp, tenors_bd=(5, 10, 15, 22, 44, 63, 126, 252), hp_p=None):
    """Term structure of vols implied by a Heston parameter set, spot-starting and forward (between consecutive tenors).

    Columns per parameter set (suffix _q for hp, _p for hp_p):
      varswap_vol  sqrt(E[int_0^T v]/T)                      closed form, = variance-swap vol
      fwd_var_vol  sqrt(E[int_{T_prev}^T v]/(T - T_prev))    closed form, exact forward variance
      atm_iv       Heston ATM-forward (K = F) implied vol    one COS call per tenor
      fwd_atm_iv   forward-variance formula applied to atm_iv
    hp_p: optional second set (e.g. real-world P parameters from fast_parameters_calibration). Missing keys
    (typically v0, which is observed and common to both measures) are taken from hp.
    Under P, varswap_vol is the expected realised vol; the gap with _q is the variance risk premium."""
    T = np.asarray(tenors_bd, float) / BDAYS_PER_YEAR
    T_prev = np.concatenate([[0.0], T[:-1]])
    out = pd.DataFrame({"tenor_bd": np.asarray(tenors_bd), "T": T})

    def term_structure(params, sfx):
        ev = expected_integrated_variance(T, params["v0"], params["kappa"], params["theta"])
        ev_prev = expected_integrated_variance(T_prev, params["v0"], params["kappa"], params["theta"])
        atm = np.array([implied_vol(heston_price(1.0, 1.0, t, 0.0, 0.0, cp=1, **params)[0], 1.0, 1.0, t, 0.0, 0.0, 1)
                        for t in T])
        atm_prev = np.concatenate([[0.0], atm[:-1]])
        out[f"varswap_vol_{sfx}"] = np.sqrt(ev / T)
        out[f"fwd_var_vol_{sfx}"] = np.sqrt((ev - ev_prev) / (T - T_prev))
        out[f"atm_iv_{sfx}"] = atm
        out[f"fwd_atm_iv_{sfx}"] = np.sqrt((T * atm**2 - T_prev * atm_prev**2) / (T - T_prev))

    term_structure(hp, "q")
    if hp_p is not None:
        term_structure({**hp, **hp_p}, "p")
        out["vrp_varswap_vol"] = out["varswap_vol_q"] - out["varswap_vol_p"]   # variance risk premium in vol points
        out["vrp_atm_iv"] = out["atm_iv_q"] - out["atm_iv_p"]
    return out


def plot_term_structure(df_ts, labels=("Q (pricing)", "P (real world)")):
    """Plotly chart of fast_forward_vol output: spot-starting (left) and forward (right) term structures.
    Colour = parameter set (Q / P), line style = measure (solid: ATM implied vol, dotted: variance-swap vol)."""
    colors = {"q": "#2a78d6", "p": "#eb6834"}            # categorical slots 1-2 (blue, orange)
    ink, muted, grid, surface = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"
    tick_bd = {5: "1W", 10: "2W", 15: "3W", 22: "1M", 44: "2M", 63: "3M", 126: "6M", 252: "1Y"}
    x = df_ts["tenor_bd"]
    sets = [s for s in ("q", "p") if f"atm_iv_{s}" in df_ts]

    fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.06,
                        subplot_titles=("Spot-starting term structure", "Forward term structure (previous tenor → tenor)"))
    for s, label in zip(("q", "p"), labels):
        if s not in sets:
            continue
        for col, (atm, var) in enumerate([(f"atm_iv_{s}", f"varswap_vol_{s}"),
                                           (f"fwd_atm_iv_{s}", f"fwd_var_vol_{s}")], start=1):
            for name, column, dash in [("ATM implied vol", atm, "solid"), ("variance-swap vol", var, "dot")]:
                fig.add_trace(go.Scatter(
                    x=x, y=df_ts[column], mode="lines+markers", name=f"{label} · {name}",
                    legendgroup=f"{s}-{name}", showlegend=(col == 1),
                    line=dict(color=colors[s], width=2, dash=dash),
                    marker=dict(size=8, color=colors[s], line=dict(color=surface, width=2)),
                    hovertemplate=f"{label} · {name}: %{{y:.2%}}<extra></extra>",
                ), row=1, col=col)

    axis = dict(type="log", tickvals=list(tick_bd), ticktext=list(tick_bd.values()), showgrid=True, gridcolor=grid,
                linecolor="#c3c2b7", tickfont=dict(color=muted), title=dict(text="tenor", font=dict(color=muted)))
    fig.update_xaxes(**axis)
    fig.update_yaxes(tickformat=".0%", showgrid=True, gridcolor=grid, linecolor="#c3c2b7", tickfont=dict(color=muted),
                     zeroline=False)
    fig.update_yaxes(title=dict(text="volatility", font=dict(color=muted)), row=1, col=1)
    fig.update_layout(
        template="plotly_white", height=480, plot_bgcolor=surface, paper_bgcolor=surface,
        font=dict(color=ink, size=13), hovermode="x unified",
        title=dict(text="Heston vol term structure: pricing (Q) vs real-world (P) parameters", x=0.01),
        legend=dict(orientation="h", y=-0.2, x=0.0, font=dict(color=ink)),
        margin=dict(l=60, r=20, t=80, b=110),
    )
    fig.update_annotations(font=dict(color="#52514e", size=13))
    return fig

## Input

In [97]:
valuation_date = pd.Timestamp("2026-11-16")   # must be a business day
holidays = []                                  # e.g. ["2026-11-26", "2026-12-25"]
S0 = 100.0
r = 0.04          # risk-free rate
q = 0.01          # dividend yield

# Heston (annualised on a 252 business-day year): low spot variance mean-reverting fast to a much higher
# long-run level -> steep upward term structure; strongly negative correlation and high vol-of-vol -> put skew.
heston_params = dict(
    v0=0.01,      # spot vol 10%
    theta=0.0625, # long-run vol 25%
    kappa=8.0,    # fast mean reversion -> the 1w-1m segment is steep
    xi=1.2,       # vol of vol
    rho=-0.75,    # spot/vol correlation
)

tenors = {"1W": pd.DateOffset(weeks=1), "2W": pd.DateOffset(weeks=2),
          "3W": pd.DateOffset(weeks=3), "1M": pd.DateOffset(months=1)}
delta_strikes = {"10P": -0.10, "20P": -0.20, "30P": -0.30, "40P": -0.40, "50C": 0.50,
                 "40C": 0.40, "30C": 0.30, "20C": 0.20, "10C": 0.10}
HORIZON_BD = 5    # horizon (business days) for the multi-day theta / breakeven

# Monte Carlo path used to back out the Heston parameters
MC_YEARS = 1      # length of the simulated path (years of business days); drives the accuracy of kappa / theta
MC_SUBSTEPS = 100   # simulation sub-steps per business day (observations are daily)
MC_SEED = 42

## Script

### Standard formulation

In [98]:
rows, rows_h = [], []
for tenor, offset in tenors.items():
    expiry = roll_following(valuation_date + offset, holidays)
    T = year_fraction_bd(valuation_date, expiry, holidays)
    for label, target in delta_strikes.items():
        cp = 1 if target > 0 else -1
        K = delta_to_strike(target, S0, T, r, q, heston_params, cp)
        g = heston_greeks(S0, K, T, r, q, heston_params, cp)
        be = heston_breakevens(g, S0, r, q, heston_params)
        keys = {"dollar_strike": K, "expiry_date": expiry.date(), "delta_strike": label, "tenor": tenor}
        rows.append({
            **keys,
            "gamma": g["gamma"], "vega": g["vega"], "vanna": g["vanna"], "volga": g["volga"],
            "theta": g["theta"] / BDAYS_PER_YEAR, "rho": g["rho"],          # theta per business day
            **{k: v / BDAYS_PER_YEAR for k, v in be.items()},              # breakevens per business day
        })
        hz = theta_breakeven_horizon(S0, K, T, r, q, heston_params, cp, HORIZON_BD)
        rows_h.append({**keys, **{f"{k}_{HORIZON_BD}bd": v for k, v in hz.items()}})

df = pd.DataFrame(rows)
df["sum_be"] = df[["be_gamma", "be_vega", "be_vanna", "be_volga", "be_rho"]].sum(axis=1)

df_horizon = pd.DataFrame(rows_h)
df_horizon[f"sum_be_{HORIZON_BD}bd"] = df_horizon[
    [f"{k}_{HORIZON_BD}bd" for k in ["be_gamma", "be_vega", "be_vanna", "be_volga", "be_rho"]]].sum(axis=1)

In [99]:
df

,dollar_strike,expiry_date,delta_strike,tenor,gamma,vega,vanna,volga,theta,rho,be_gamma,be_vega,be_vanna,be_volga,be_rho,sum_be
0,97.409036,2026-11-23,10P,1W,0.046992,8.296075,-4.209616,30.502030,-0.038215,-0.139651,-0.009324,-0.013827,-0.015034,-0.000871,0.000842,-0.038215
1,98.483596,2026-11-23,20P,1W,0.094779,14.021274,-5.909036,-188.293388,-0.056146,-0.290519,-0.018805,-0.023369,-0.021104,0.005380,0.001752,-0.056146
2,99.152375,2026-11-23,30P,1W,0.144205,18.263757,-6.076026,-426.625820,-0.065841,-0.451252,-0.028612,-0.030440,-0.021700,0.012189,0.002721,-0.065841
3,99.655444,2026-11-23,40P,1W,0.196200,21.187282,-4.780884,-619.475135,-0.069856,-0.623601,-0.038929,-0.035312,-0.017075,0.017699,0.003761,-0.069856
4,100.071217,2026-11-23,50C,1W,0.250689,22.742822,-1.894110,-710.228646,-0.081135,1.173522,-0.049740,-0.037905,-0.006765,0.020292,-0.007017,-0.081134
5,100.441204,2026-11-23,40C,1W,0.304185,22.736968,2.743962,-632.748558,-0.076202,0.974318,-0.060354,-0.037895,0.009800,0.018079,-0.005829,-0.076200
6,100.794066,2026-11-23,30C,1W,0.342732,20.814810,8.820426,-318.648551,-0.066583,0.750716,-0.068002,-0.034691,0.031502,0.009104,-0.004494,-0.066582
7,101.166586,2026-11-23,20C,1W,0.332136,16.510559,14.365440,222.715528,-0.051478,0.501578,-0.065900,-0.027518,0.051305,-0.006363,-0.003004,-0.051479
8,101.648507,2026-11-23,10C,1W,0.219102,9.461226,13.947901,667.246117,-0.029911,0.237300,-0.043473,-0.015769,0.049814,-0.019064,-0.001422,-0.029913
9,95.888711,2026-11-30,10P,2W,0.026701,9.968367,-2.914521,-6.963994,-0.031315,-0.266796,-0.005298,-0.016614,-0.010409,0.000199,0.000807,-0.031315


In [100]:
df_horizon

,dollar_strike,expiry_date,delta_strike,tenor,theta_5bd,be_gamma_5bd,be_vega_5bd,be_vanna_5bd,be_volga_5bd,be_rho_5bd,sum_be_5bd
0,97.409036,2026-11-23,10P,1W,-0.100898,-0.025319,-0.026307,-0.041117,-0.009826,0.001670,-0.100898
1,98.483596,2026-11-23,20P,1W,-0.214991,-0.076425,-0.058114,-0.085747,0.000565,0.004731,-0.214991
2,99.152375,2026-11-23,30P,1W,-0.339586,-0.163618,-0.091416,-0.121173,0.027213,0.009408,-0.339587
3,99.655444,2026-11-23,40P,1W,-0.475539,-0.313980,-0.121803,-0.122268,0.066072,0.016440,-0.475538
4,100.071217,2026-11-23,50C,1W,-0.613529,-0.516698,-0.138913,-0.019959,0.094351,-0.032298,-0.613518
5,100.441204,2026-11-23,40C,1W,-0.413161,-0.453387,-0.129938,0.127124,0.064706,-0.021654,-0.413149
6,100.794066,2026-11-23,30C,1W,-0.260297,-0.361020,-0.100718,0.221393,-0.006868,-0.013089,-0.260301
7,101.166586,2026-11-23,20C,1W,-0.143979,-0.228564,-0.062087,0.212711,-0.059464,-0.006590,-0.143993
8,101.648507,2026-11-23,10C,1W,-0.057642,-0.091649,-0.025509,0.115770,-0.054025,-0.002232,-0.057644
9,95.888711,2026-11-30,10P,2W,-0.130863,-0.022491,-0.057259,-0.046936,-0.006952,0.002775,-0.130863


In [101]:
be_cols = ["be_gamma", "be_vega", "be_vanna", "be_volga", "be_rho"]
residual = df[be_cols].sum(axis=1) - df["theta"]
print(f"1 business day : max |sum(be) - theta| = {residual.abs().max():.2e}  (max |theta| = {df['theta'].abs().max():.4f})")

be_cols_h = [f"{c}_{HORIZON_BD}bd" for c in be_cols]
residual_h = df_horizon[be_cols_h].sum(axis=1) - df_horizon[f"theta_{HORIZON_BD}bd"]
print(f"{HORIZON_BD} business days: max |sum(be) - theta| = {residual_h.abs().max():.2e}  (max |theta| = {df_horizon[f'theta_{HORIZON_BD}bd'].abs().max():.4f})")

1 business day : max |sum(be) - theta| = 2.44e-06  (max |theta| = 0.0811)
5 business days: max |sum(be) - theta| = 1.34e-05  (max |theta| = 0.6135)


### Forward formulation

In [102]:
rows_f = []
for _, row in df.iterrows():
    expiry = pd.Timestamp(row["expiry_date"])
    T = year_fraction_bd(valuation_date, expiry, holidays)
    K, cp = row["dollar_strike"], (1 if row["delta_strike"].endswith("C") else -1)
    F = S0 * np.exp((r - q) * T)
    g = heston_greeks_fwd(F, K, T, r, heston_params, cp)
    be = heston_breakevens_fwd(g, F, heston_params)
    rows_f.append({
        "dollar_strike": K, "expiry_date": row["expiry_date"], "delta_strike": row["delta_strike"],
        "tenor": row["tenor"], "forward": F,
        "gamma": g["gamma"], "vega": g["vega"], "vanna": g["vanna"], "volga": g["volga"],
        "theta": g["theta"] / BDAYS_PER_YEAR,                              # forward theta per business day
        **{k: v / BDAYS_PER_YEAR for k, v in be.items()},                  # breakevens per business day
    })

df_fwd = pd.DataFrame(rows_f)
df_fwd["sum_be"] = df_fwd[["be_gamma", "be_vega", "be_vanna", "be_volga"]].sum(axis=1)

In [103]:
df_fwd

,dollar_strike,expiry_date,delta_strike,tenor,forward,gamma,vega,vanna,volga,theta,be_gamma,be_vega,be_vanna,be_volga,sum_be
0,97.409036,2026-11-23,10P,1W,100.059542,0.046936,8.296075,-4.207111,30.502031,-0.039056,-0.009324,-0.013827,-0.015034,-0.000871,-0.039056
1,98.483596,2026-11-23,20P,1W,100.059542,0.094666,14.021274,-5.905519,-188.293388,-0.057898,-0.018805,-0.023369,-0.021104,0.005380,-0.057898
2,99.152375,2026-11-23,30P,1W,100.059542,0.144033,18.263757,-6.072411,-426.625820,-0.068562,-0.028612,-0.030440,-0.021700,0.012189,-0.068562
3,99.655444,2026-11-23,40P,1W,100.059542,0.195967,21.187282,-4.778039,-619.475134,-0.073616,-0.038929,-0.035312,-0.017075,0.017699,-0.073616
4,100.071217,2026-11-23,50C,1W,100.059542,0.250391,22.742822,-1.892983,-710.228646,-0.074118,-0.049740,-0.037905,-0.006765,0.020292,-0.074117
5,100.441204,2026-11-23,40C,1W,100.059542,0.303824,22.736968,2.742329,-632.748559,-0.070372,-0.060354,-0.037895,0.009800,0.018079,-0.070371
6,100.794066,2026-11-23,30C,1W,100.059542,0.342324,20.814810,8.815177,-318.648552,-0.062090,-0.068002,-0.034691,0.031502,0.009104,-0.062088
7,101.166586,2026-11-23,20C,1W,100.059542,0.331740,16.510559,14.356892,222.715528,-0.048475,-0.065900,-0.027518,0.051305,-0.006363,-0.048476
8,101.648507,2026-11-23,10C,1W,100.059542,0.218841,9.461226,13.939601,667.246118,-0.028489,-0.043473,-0.015769,0.049814,-0.019064,-0.028492
9,95.888711,2026-11-30,10P,2W,100.119119,0.026638,9.968367,-2.911053,-6.963994,-0.032122,-0.005298,-0.016614,-0.010409,0.000199,-0.032122


In [104]:
be_cols_f = ["be_gamma", "be_vega", "be_vanna", "be_volga"]
print(f"forward: max |sum(be) - theta|              = {(df_fwd['sum_be'] - df_fwd['theta']).abs().max():.2e}")
print(f"forward vs spot: max |be_i(fwd) - be_i(spot)| = {(df_fwd[be_cols_f] - df[be_cols_f]).abs().max().max():.2e}")
print(f"forward vs spot: max |theta_fwd - (theta - be_rho)| = {(df_fwd['theta'] - (df['theta'] - df['be_rho'])).abs().max():.2e}")

forward: max |sum(be) - theta|              = 2.38e-06
forward vs spot: max |be_i(fwd) - be_i(spot)| = 1.27e-10
forward vs spot: max |theta_fwd - (theta - be_rho)| = 5.75e-08


### Monte Carlo path and parameter estimation

In [105]:
S_path, v_path = simulate_heston_path(S0, r, q, heston_params, MC_YEARS, MC_SUBSTEPS, MC_SEED)
estimated = estimate_heston_params(S_path, v_path, 1.0 / BDAYS_PER_YEAR, r, q)

df_estimation = pd.DataFrame({
    "parameter": ["long-term variance (theta)", "speed of mean reversion (kappa)",
                  "vol of vol (xi)", "spot/variance correlation (rho)"],
    "true": [heston_params[k] for k in estimated],
    "estimated": list(estimated.values()),
})
df_estimation["error"] = df_estimation["estimated"] - df_estimation["true"]
df_estimation["relative_error"] = df_estimation["error"] / df_estimation["true"]
print(f"path: {len(v_path) - 1} business days, share of days with v = 0: {(v_path == 0).mean():.2%}")
df_estimation

path: 252 business days, share of days with v = 0: 0.00%


,parameter,true,estimated,error,relative_error
0,long-term variance (theta),0.062500,0.045500,-0.017000,-0.272004
1,speed of mean reversion (kappa),8.000000,6.292939,-1.707061,-0.213383
2,vol of vol (xi),1.200000,1.150048,-0.049952,-0.041627
3,spot/variance correlation (rho),-0.750000,-0.776208,-0.026208,0.034944


### Forward implied volatility: 3W in 1W

In [106]:
short_tenor, long_tenor = "1W", "1M"
t_exp = pd.Timestamp(df.loc[df["tenor"] == short_tenor, "expiry_date"].iloc[0])
T_exp = pd.Timestamp(df.loc[df["tenor"] == long_tenor, "expiry_date"].iloc[0])
t = year_fraction_bd(valuation_date, t_exp, holidays)
T = year_fraction_bd(valuation_date, T_exp, holidays)

rows_fwd_vol = []
for label, target in delta_strikes.items():
    cp = 1 if target > 0 else -1
    K_t = df.loc[(df["tenor"] == short_tenor) & (df["delta_strike"] == label), "dollar_strike"].iloc[0]
    K_T = df.loc[(df["tenor"] == long_tenor) & (df["delta_strike"] == label), "dollar_strike"].iloc[0]
    iv_t = implied_vol(heston_price(S0, K_t, t, r, q, cp=cp, **heston_params)[0], S0, K_t, t, r, q, cp)
    iv_T = implied_vol(heston_price(S0, K_T, T, r, q, cp=cp, **heston_params)[0], S0, K_T, T, r, q, cp)
    iv_fwd_formula = np.sqrt((T * iv_T**2 - t * iv_t**2) / (T - t))
    k_fwd, iv_fwd_heston = forward_start_iv_at_delta(target, t, T - t, r, q, heston_params, cp)
    rows_fwd_vol.append({
        "delta_strike": label,
        f"iv_{short_tenor}": iv_t,
        f"iv_{long_tenor}": iv_T,
        "fwd_iv_formula": iv_fwd_formula,
        "fwd_iv_heston": iv_fwd_heston,
        "fwd_relative_strike": k_fwd,
        "diff_vol_pts": 100 * (iv_fwd_formula - iv_fwd_heston),
    })

df_fwd_vol = pd.DataFrame(rows_fwd_vol)
print(f"t = {short_tenor} ({t * BDAYS_PER_YEAR:.0f} bd), T = {long_tenor} ({T * BDAYS_PER_YEAR:.0f} bd), "
      f"forward period = {(T - t) * BDAYS_PER_YEAR:.0f} bd")
df_fwd_vol

t = 1W (5 bd), T = 1M (22 bd), forward period = 17 bd


,delta_strike,iv_1W,iv_1M,fwd_iv_formula,fwd_iv_heston,fwd_relative_strike,diff_vol_pts
0,10P,0.149968,0.204168,0.217555,0.214490,0.934449,0.306536
1,20P,0.135472,0.179879,0.190984,0.188710,0.962710,0.227390
2,30P,0.125453,0.163829,0.173508,0.171780,0.979854,0.172812
3,40P,0.117319,0.151358,0.159997,0.158759,0.992484,0.123840
4,50C,0.110231,0.141012,0.148859,0.148167,1.002735,0.069227
5,40C,0.103773,0.131905,0.139100,0.139291,1.011889,-0.019031
6,30C,0.097765,0.123758,0.130421,0.132151,1.020806,-0.173066
7,20C,0.092119,0.116396,0.122625,0.127153,1.030814,-0.452848
8,10C,0.086872,0.109950,0.115866,0.125447,1.045294,-0.958077


### Usage example: fast calibration functions

**`fast_options_calibration(market)`** calibrates the Heston pricing ($\mathbb{Q}$) parameters to option PVs.
- **Input columns:** `strike`, `time_to_maturity` (business-day years), `forward`, `discount`, `pv`, and optionally `cp` (+1 call / −1 put). If `cp` is missing, the out-of-the-money option is assumed: a call if strike ≥ forward, otherwise a put.
- **Objective:** least squares on $(C^{\text{model}} - C^{\text{mkt}})/\text{Vega}^{\text{Black}}$, i.e. approximately implied-vol errors. Each maturity is priced in one vectorised COS call using $C(F, K) = F\,C(1, K/F)$, so a calibration takes a fraction of a second.
- **Output:** the fitted $(v_0, \bar\theta, \kappa, \xi, \rho)$, and the input dataframe plus `pv_heston`. It also adds the forward-formulation Greeks (in PV terms) and the four breakevens with `sum_be` = `theta` (per business day). The inputs are forwards and discount factors, so there is no carry term.
- **Identification:** with maturities between 1W and 1M only, $v_0$, $\xi$ and $\rho$ are well identified, but $\kappa$ and $\bar\theta$ trade off against each other. Adding longer maturities, or fixing one of them, stabilises the fit.

**`fast_parameters_calibration(variance, forward)`** estimates the real-world ($\mathbb{P}$) parameters $(\bar\theta, \kappa, \xi, \rho)$ from time series of instantaneous variance and a forward. It uses the same estimators as the Monte Carlo section. The forward log-returns are de-meaned because the $\mathbb{P}$ drift is unknown.

**Is the 1W forward a good proxy for the spot to estimate the spot/variance correlation?** Yes.
- $\ln F = \ln S + (r - q)\tau$. Daily changes in $(r - q)\tau$ are deterministic or tiny next to equity moves, so $d\ln F \approx d\ln S$. With deterministic rates, the forward and the spot share the same Brownian motion, so $\rho$ is identical.
- The forward is often better than the spot: it does not jump at discrete ex-dividend dates.
- Compute each return on the same contract (no roll jumps), or use a constant-maturity interpolated forward.
- On real data, $v$ is not observed. A natural proxy is the daily $v_0$ from `fast_options_calibration`, or a short-dated ATM implied variance. Comparing $\xi$, $\rho$ under $\mathbb{P}$ and $\mathbb{Q}$ is a consistency check: they should agree. The gap in $\kappa$, $\bar\theta$ reflects the variance risk premium.

The examples below are round trips. The options are priced with the notebook's Heston parameters and calibrated back from a generic starting point. The time series is the simulated Monte Carlo path, with the 1W forward $F = S\,e^{(r-q)\cdot 5/252}$.

In [107]:
# 1) options calibration: market quotes = the notebook's 1W-1M delta grid priced with heston_params
market = pd.DataFrame({"strike": df["dollar_strike"]})
market["time_to_maturity"] = [year_fraction_bd(valuation_date, pd.Timestamp(e), holidays) for e in df["expiry_date"]]
market["forward"] = S0 * np.exp((r - q) * market["time_to_maturity"])
market["discount"] = np.exp(-r * market["time_to_maturity"])
market["cp"] = np.where(df["delta_strike"].str.endswith("C"), 1, -1)
market["pv"] = [heston_price(S0, K, T, r, q, cp=cp, **heston_params)[0]
                for K, T, cp in zip(market["strike"], market["time_to_maturity"], market["cp"])]

fitted_q, df_calibrated = fast_options_calibration(market, x0=[0.04, 0.04, 1.0, 0.3, 0.0])
print(pd.DataFrame({"true": heston_params, "fitted": fitted_q}).T)
print(f"max |pv_heston - pv| = {(df_calibrated['pv_heston'] - df_calibrated['pv']).abs().max():.2e},  "
      f"max |sum_be - theta| = {(df_calibrated['sum_be'] - df_calibrated['theta']).abs().max():.2e}")
df_calibrated

             v0    theta    kappa       xi       rho
true   0.010000 0.062500 8.000000 1.200000 -0.750000
fitted 0.010000 0.062500 8.000000 1.200000 -0.750000
max |pv_heston - pv| = 1.02e-13,  max |sum_be - theta| = 2.38e-06


,strike,time_to_maturity,forward,discount,cp,pv,pv_heston,delta,gamma,vega,vanna,volga,theta,be_gamma,be_vega,be_vanna,be_volga,sum_be
0,97.409036,0.019841,100.059542,0.999207,-1,0.100898,0.100898,-0.069334,0.046936,8.296075,-4.207111,30.502032,-0.039056,-0.009324,-0.013827,-0.015034,-0.000871,-0.039056
1,98.483596,0.019841,100.059542,0.999207,-1,0.214991,0.214991,-0.144187,0.094666,14.021274,-5.905519,-188.293387,-0.057898,-0.018805,-0.023369,-0.021104,0.005380,-0.057898
2,99.152375,0.019841,100.059542,0.999207,-1,0.339586,0.339586,-0.223903,0.144033,18.263757,-6.072411,-426.625820,-0.068562,-0.028612,-0.030440,-0.021700,0.012189,-0.068562
3,99.655444,0.019841,100.059542,0.999207,-1,0.475539,0.475539,-0.309357,0.195967,21.187282,-4.778039,-619.475134,-0.073616,-0.038929,-0.035312,-0.017075,0.017699,-0.073616
4,100.071217,0.019841,100.059542,0.999207,1,0.613529,0.613529,0.597232,0.250391,22.742822,-1.892983,-710.228649,-0.074118,-0.049740,-0.037905,-0.006765,0.020292,-0.074117
5,100.441204,0.019841,100.059542,0.999207,1,0.413161,0.413161,0.494891,0.303824,22.736968,2.742329,-632.748557,-0.070372,-0.060354,-0.037895,0.009800,0.018079,-0.070371
6,100.794066,0.019841,100.059542,0.999207,1,0.260297,0.260297,0.380736,0.342324,20.814810,8.815177,-318.648550,-0.062090,-0.068002,-0.034691,0.031502,0.009104,-0.062088
7,101.166586,0.019841,100.059542,0.999207,1,0.143979,0.143979,0.254086,0.331740,16.510559,14.356892,222.715527,-0.048475,-0.065900,-0.027518,0.051305,-0.006363,-0.048476
8,101.648507,0.019841,100.059542,0.999207,1,0.057642,0.057642,0.120109,0.218841,9.461226,13.939601,667.246113,-0.028489,-0.043473,-0.015769,0.049814,-0.019064,-0.028492
9,95.888711,0.039683,100.119119,0.998414,-1,0.164006,0.164006,-0.065515,0.026638,9.968367,-2.911053,-6.963993,-0.032122,-0.005298,-0.016614,-0.010409,0.000199,-0.032122


In [108]:
# 2) real-world parameters from the simulated path: instantaneous variance + 1W forward as spot proxy
forward_1w = S_path * np.exp((r - q) * 5 / BDAYS_PER_YEAR)
fitted_p = fast_parameters_calibration(v_path, forward_1w)
print(f"path length: {len(v_path) - 1} business days (MC_YEARS = {MC_YEARS})")
pd.DataFrame({"true": {k: heston_params[k] for k in fitted_p}, "fitted (P)": fitted_p}).T

path length: 252 business days (MC_YEARS = 1)


,theta,kappa,xi,rho
true,0.062500,8.000000,1.200000,-0.750000
fitted (P),0.045500,6.292939,1.150048,-0.776208


### Usage example: `fast_forward_vol`
**`fast_forward_vol(hp, tenors_bd, hp_p=None)`** returns the vol term structure implied by a Heston parameter set, on a grid of business-day tenors. For each tenor $T$ (previous tenor $T_-$) it gives:

| column | definition | how |
|---|---|---|
| `varswap_vol` | $\sqrt{\mathbb{E}\big[\int_0^T v\,ds\big]/T}$ | closed form: $\mathbb{E}\big[\int_0^T v\big] = \bar\theta T + (v_0 - \bar\theta)\dfrac{1 - e^{-\kappa T}}{\kappa}$ |
| `fwd_var_vol` | $\sqrt{\mathbb{E}\big[\int_{T_-}^{T} v\,ds\big]/(T - T_-)}$ | closed form, exact forward variance |
| `atm_iv` | Heston ATM-forward ($K = F$) implied vol | one COS call per tenor |
| `fwd_atm_iv` | $\sqrt{(T\,\sigma_T^2 - T_-\,\sigma_{T_-}^2)/(T - T_-)}$ on `atm_iv` | forward-variance formula |

Columns end in `_q` for `hp`. If a second set `hp_p` is given (e.g. the real-world output of `fast_parameters_calibration`), the same columns are computed with suffix `_p`. Missing keys, typically $v_0$, which is observed and common to both measures, are taken from `hp`. Two comparison columns are added: `vrp_varswap_vol` = `varswap_vol_q` − `varswap_vol_p` and `vrp_atm_iv` = `atm_iv_q` − `atm_iv_p`.

Under $\mathbb{P}$, `varswap_vol_p` is the **expected realised vol**. The gap with the $\mathbb{Q}$ variance-swap vol is the **variance risk premium** term structure. `atm_iv_p` is the ATM vol the option would have if it were priced with the $\mathbb{P}$ parameters.

In the example, `hp_p` is the $\mathbb{P}$ estimate from the simulated path. The path was simulated with `heston_params`, so the true premium is zero, and the gap shown is pure estimation noise. It shrinks as `MC_YEARS` increases.

In [109]:
df_term_structure = fast_forward_vol(heston_params, tenors_bd=(5, 10, 15, 22, 44, 63, 126, 252), hp_p=fitted_p)
df_term_structure

,tenor_bd,T,varswap_vol_q,fwd_var_vol_q,atm_iv_q,fwd_atm_iv_q,varswap_vol_p,fwd_var_vol_p,atm_iv_p,fwd_atm_iv_p,vrp_varswap_vol,vrp_atm_iv
0,5,0.019841,0.118130,0.118130,0.110433,0.110433,0.110122,0.110122,0.101923,0.101923,0.008008,0.008510
1,10,0.039683,0.132353,0.145189,0.120996,0.130707,0.118682,0.126665,0.105748,0.109439,0.013671,0.015248
2,15,0.059524,0.143984,0.164801,0.130492,0.147663,0.126059,0.139648,0.110316,0.118927,0.017925,0.020176
3,22,0.087302,0.157218,0.182371,0.141783,0.163373,0.134857,0.152004,0.116613,0.129078,0.022361,0.025170
4,44,0.174603,0.184967,0.209064,0.166445,0.187896,0.154787,0.172428,0.132649,0.146945,0.030180,0.033795
5,63,0.250000,0.199506,0.229668,0.179898,0.207736,0.166175,0.189945,0.142558,0.163211,0.033330,0.037340
6,126,0.500000,0.222745,0.243779,0.203155,0.224010,0.186286,0.204428,0.161892,0.179151,0.036459,0.041263
7,252,1.000000,0.236516,0.249528,0.219955,0.235559,0.199672,0.212215,0.178156,0.193055,0.036844,0.041798


In [110]:
fig_term_structure = plot_term_structure(df_term_structure)
fig_term_structure.show()